## Inspecting the `data` DataFrame

In [3]:
# Flatten MultiIndex columns if they exist
if isinstance(data.columns, pd.MultiIndex):
    # Prioritize 'Price' level if it exists and is meaningful
    if 'Price' in data.columns.names:
        data.columns = data.columns.get_level_values('Price')
    else: # Fallback to the first level if 'Price' isn't explicitly named
        data.columns = data.columns.get_level_values(0)

print('Columns of the DataFrame after flattening:')
print(data.columns)
print('\nFirst 5 rows of the DataFrame after flattening:')
display(data.head())

NameError: name 'data' is not defined

## Handling NaN Values and Explaining Features

In [ ]:
# Check for NaN values introduced by feature engineering
print('Number of NaN values before dropping:')
print(data.isnull().sum())

# Drop rows with NaN values
data.dropna(inplace=True)

print('\nNumber of NaN values after dropping:')
print(data.isnull().sum())
print(f'\nShape of DataFrame after dropping NaNs: {data.shape}')

# Display the first few rows after cleaning
print('\nFirst 5 rows of the DataFrame after handling NaNs:')
display(data.head())

### Explanation of Engineered Features:

**1. Moving Averages (MA_20, MA_50):**
   - **Purpose:** Moving averages smooth out price data over a specified period, helping to identify trends by reducing random short-term fluctuations. They are lagging indicators.
   - **MA_20 (20-day Moving Average):** Represents the average closing price over the last 20 trading days. It's often used to identify short-term trends and potential support/resistance levels.
   - **MA_50 (50-day Moving Average):** Represents the average closing price over the last 50 trading days. It provides a more stable, medium-term trend indication. A common strategy involves comparing MA_20 and MA_50 to generate buy/sell signals (e.g., a 'golden cross' when MA_20 crosses above MA_50, or a 'death cross' when it crosses below).

**2. Daily Return (Daily_Return):**
   - **Purpose:** Measures the percentage change in price from one day to the next. It quantifies the profitability or loss of an investment over a single day.
   - **Usefulness:** Daily returns are fundamental for understanding price volatility and can be used as features to predict future price movements or trends. They normalize price changes, making them comparable across different stocks and time periods.

**3. Volatility (Volatility):**
   - **Purpose:** Measures the degree of variation of a trading price series over time. It quantifies how much the price of an asset fluctuates.
   - **Calculation:** Typically calculated as the standard deviation of daily returns over a period (e.g., 20 days). A higher volatility value indicates larger price swings, while a lower value suggests more stable prices.
   - **Usefulness:** Volatility is crucial for risk assessment. High volatility might indicate higher risk but also potential for higher returns, while low volatility suggests lower risk. It can be used by models to gauge market sentiment and risk appetite.

## Exploratory Data Analysis (EDA)

EDA is a critical step to understand the characteristics of our data, identify patterns, and detect anomalies. We will visualize key aspects of the historical stock data and the features we've engineered.

### 1. Closing Price and Moving Averages Over Time

This plot shows the historical closing price of NIFTY 50 along with its 20-day and 50-day moving averages. This visualization helps in observing the general trend, as well as how the shorter-term MA_20 interacts with the longer-term MA_50, which can often signal shifts in momentum.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Closing Price', alpha=0.7)
plt.plot(data['MA_20'], label='20-Day Moving Average', alpha=0.7)
plt.plot(data['MA_50'], label='50-Day Moving Average', alpha=0.7)
plt.title('NIFTY 50 Closing Price and Moving Averages Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

### 2. Daily Return Distribution

This histogram visualizes the distribution of `Daily_Return`. It helps us understand the frequency of different return magnitudes and identify common patterns, such as whether returns are normally distributed or exhibit fat tails (more extreme gains/losses than a normal distribution would suggest).

In [4]:
plt.figure(figsize=(12, 6))
sns.histplot(data['Daily_Return'].dropna(), bins=50, kde=True)
plt.title('Distribution of Daily Returns for NIFTY 50')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

NameError: name 'plt' is not defined

### 3. Volatility Over Time

This plot displays the calculated `Volatility` (rolling standard deviation of daily returns) over time. High volatility indicates periods of significant price swings and higher perceived risk, while low volatility suggests more stable market conditions. Observing this trend helps in understanding the market's risk profile.

In [5]:
plt.figure(figsize=(14, 7))
plt.plot(data['Volatility'], label='20-Day Rolling Volatility', color='purple', alpha=0.7)
plt.title('NIFTY 50 Volatility Over Time')
plt.xlabel('Date')
plt.ylabel('Volatility (Standard Deviation of Daily Returns)')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'plt' is not defined

### 4. Correlation Analysis

Correlation analysis helps us understand the linear relationships between different variables in our dataset. A heatmap is an excellent way to visualize this, where each cell represents the correlation coefficient between two variables. Values closer to 1 or -1 indicate a strong positive or negative correlation, respectively, while values closer to 0 suggest a weak or no linear correlation.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Features')
plt.show()

### 5. Autocorrelation Analysis

Autocorrelation measures how much a time series is correlated with its past values. For stock prices, significant autocorrelation indicates that past prices can provide information about future prices. The Autocorrelation Function (ACF) plot shows the correlation coefficients between the series and its lagged values. This is particularly useful for identifying the order of the AR (AutoRegressive) component in ARIMA models.

In [6]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(12, 6))
plot_acf(data['Close'], lags=50, ax=plt.gca(), title='Autocorrelation Function (ACF) of NIFTY 50 Closing Price')
plt.xlabel('Lags')
plt.ylabel('Autocorrelation')
plt.grid(True)
plt.show()

NameError: name 'plt' is not defined

### 6. Stationarity Analysis (Augmented Dickey-Fuller Test)

Stationarity is a key assumption for many time series models, including ARIMA. A stationary time series has statistical properties (like mean, variance, and autocorrelation) that are constant over time. Non-stationary series are often transformed (e.g., by differencing) to become stationary.

The Augmented Dickey-Fuller (ADF) test is a common statistical test to check for a unit root in a time series, which indicates non-stationarity. The null hypothesis (H0) of the ADF test is that the time series has a unit root (i.e., it is non-stationary). The alternative hypothesis (H1) is that the series is stationary.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def test_stationarity(timeseries):
    # Determing rolling statistics
    rolmean = timeseries.rolling(12).mean()
    rolstd = timeseries.rolling(12).std()

    # Plot rolling statistics:
    plt.figure(figsize=(14, 7))
    plt.plot(timeseries, color='blue', label='Original', alpha=0.7)
    plt.plot(rolmean, color='red', label='Rolling Mean', alpha=0.7)
    plt.plot(rolstd, color='black', label='Rolling Std', alpha=0.7)
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.grid(True)
    plt.show()

    # Perform Dickey-Fuller test:
    # print('Results of Augmented Dickey-Fuller Test:') # This will be printed by the calling cell
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic', 'p-value', '#Lags Used', 'Number of Observations Used'])
    for key, value in dftest[4].items():
        dfoutput[f'Critical Value ({key})'] = value
    # print(dfoutput) # This will be printed by the calling cell

    conclusion_string = ""
    if dfoutput['p-value'] <= 0.05:
        conclusion_string = "\nConclusion: The p-value is less than or equal to 0.05. We reject the null hypothesis, which means the series is stationary."
    else:
        conclusion_string = "\nConclusion: The p-value is greater than 0.05. We fail to reject the null hypothesis, which means the series is non-stationary and has a unit root."

    return dfoutput, conclusion_string

# Test stationarity on the 'Close' price (original call will now print the results)
print('Results of Augmented Dickey-Fuller Test on Close Price:')
original_adf_results, original_conclusion = test_stationarity(data['Close'])
print(original_adf_results)
print(original_conclusion)


### Addressing Non-Stationarity: Differencing

Since the ADF test indicated non-stationarity, we need to transform the series. A common approach is to apply **differencing**, which involves computing the difference between consecutive observations. First-order differencing is often sufficient to achieve stationarity for financial time series.

In [ ]:
# Apply first-order differencing to the 'Close' price
data['Differenced_Close'] = data['Close'].diff()

print('First 5 rows of the Differenced_Close series (with potential NaN):')
display(data['Differenced_Close'].head())

# Re-test stationarity on the differenced series
print('\nRe-testing stationarity on the Differenced_Close series:')
print('Results of Augmented Dickey-Fuller Test on Differenced_Close Price:')
differenced_adf_results, differenced_conclusion = test_stationarity(data['Differenced_Close'].dropna())
print(differenced_adf_results)
print(differenced_conclusion)


In [ ]:
# 7. BUILDING FEATURES AND TARGETS FOR ML

import numpy as np
import pandas as pd

data['Target_Close_Next'] = data['Close'].shift(-1)
data['Trend_Next'] = (data['Close'].shift(-1) > data['Close']).astype(int)

feature_cols = ['Open', 'High', 'Low', 'Volume', 'MA_20', 'MA_50', 'Daily_Return', 'Volatility']

ml_data = data.dropna(subset=['Target_Close_Next', 'Trend_Next']).copy()

print("Total rows for ML:", len(ml_data))
print("Date range:", ml_data.index.min().date(), "to", ml_data.index.max().date())

print("\nTrend_Next class balance:")
print(ml_data['Trend_Next'].value_counts(normalize=True).round(3))

split_idx = int(len(ml_data) * 0.80)
train = ml_data.iloc[:split_idx]
test = ml_data.iloc[split_idx:]

print("\nTrain size:", len(train), "| Test size:", len(test))
print("Train:", train.index.min().date(), "to", train.index.max().date())
print("Test :", test.index.min().date(), "to", test.index.max().date())

X_train = train[feature_cols]
X_test = test[feature_cols]

y_reg_train = train['Target_Close_Next']
y_reg_test = test['Target_Close_Next']

y_clf_train = train['Trend_Next']
y_clf_test = test['Trend_Next']

print("\nX_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)